# SwiftSQL V2: 真实沙箱评测（Base vs V1）+ 执行引导 DPO 对齐 + V2 导出

> **核心思想**（参考 Auto-ThinkSQL & Execution-Guided RL）：
> 1. 先在独立验证集上对比 **未微调基座 3B vs V1 SFT**（看真实 SQLite 执行准确率 EX、语法率、拒答 F1、输出长度与耗时）；
> 2. 用 V1 对带真实表的训练集采样多条候选 SQL，放进 SQLite 沙箱**真正跑一遍**，自动筛选 `Chosen = 执行正确且简洁`、`Rejected = 执行报错 / 结果不对 / 啰嗦`；
> 3. 跑一轮 Unsloth **DPO**，让模型同时变得**更准、更短、更快**，并再次评测验证提升。


In [ ]:
import os, time, torch, yaml
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/2422725977/swift-sql.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
import sys
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

with open(f"{REPO_DIR}/configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)


In [ ]:
# 1. 定义统一推理与评测函数（对比 Base vs V1 SFT）
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from swiftsql.data_pipeline import load_jsonl, save_jsonl
from swiftsql.evaluation import evaluate_predictions

EVAL_PATH = Path(DRIVE_DIR) / "data" / "eval.jsonl"
TRAIN_PATH = Path(DRIVE_DIR) / "data" / "train_sft.jsonl"
eval_records = load_jsonl(EVAL_PATH)
train_records = load_jsonl(TRAIN_PATH)
print(f"Loaded eval={len(eval_records)}, train={len(train_records)}")

def run_benchmark(model_name_or_path, records, max_samples=200):
    subset = records[:max_samples]
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(m)
    preds, latencies = [], []
    for rec in tqdm(subset, desc=f"Eval {Path(model_name_or_path).name}"):
        prompt_msgs = [x for x in rec["messages"] if x["role"] != "assistant"]
        prompt = tok.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok([prompt], return_tensors="pt").to("cuda")
        t0 = time.perf_counter()
        out = m.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=0.0)
        latencies.append((time.perf_counter() - t0) * 1000.0)
        gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        preds.append(gen)
    report = evaluate_predictions(subset, preds, latencies_ms=latencies)
    report.pop("details", None)
    del m
    torch.cuda.empty_cache()
    return report

base_report = run_benchmark(CFG["model"]["base_model"], eval_records, max_samples=200)
v1_report = run_benchmark(f"{DRIVE_DIR}/models/v1_sft_lora", eval_records, max_samples=200)

print("\n=== Benchmark Comparison: Base vs V1 SFT ===")
for k in ["json_valid_rate", "syntax_valid_rate", "execution_accuracy", "refusal_f1", "avg_output_chars", "avg_latency_ms"]:
    print(f"{k:22s} | Base: {str(base_report.get(k)):>8s} | V1 SFT: {str(v1_report.get(k)):>8s}")


In [ ]:
# 2. 离线生成执行验证 DPO 偏好对（运行真实 SQLite 沙箱筛选“对且短”vs“错或啰嗦”）
from swiftsql.dpo_builder import build_dpo_pair

DPO_PATH = Path(DRIVE_DIR) / "data" / "dpo_pairs.jsonl"
dpo_cfg = CFG["dpo"]

if DPO_PATH.exists():
    dpo_pairs = load_jsonl(DPO_PATH)
    print(f"Loaded cached DPO pairs: {len(dpo_pairs)}")
else:
    exec_pool = [r for r in train_records if r["meta"].get("executable_context")]
    refusal_pool = [r for r in train_records if r["task_type"] == "refusal"][:150]
    pool = exec_pool[: dpo_cfg["num_prompts"]] + refusal_pool

    m, tok = FastLanguageModel.from_pretrained(
        model_name=f"{DRIVE_DIR}/models/v1_sft_lora",
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(m)

    dpo_pairs = []
    n_cand = dpo_cfg["candidates_per_prompt"]
    for rec in tqdm(pool, desc="Sampling & executing candidates"):
        prompt_msgs = [x for x in rec["messages"] if x["role"] != "assistant"]
        prompt = tok.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok([prompt], return_tensors="pt").to("cuda")
        outs = m.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=n_cand,
        )
        cands = [
            tok.decode(seq[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            for seq in outs
        ]
        pair = build_dpo_pair(rec, cands)
        if pair is not None:
            dpo_pairs.append(pair)

    save_jsonl(dpo_pairs, DPO_PATH)
    del m
    torch.cuda.empty_cache()
    print(f"Built {len(dpo_pairs)} execution-verified DPO pairs -> {DPO_PATH}")


In [ ]:
# 3. 启动 V2 DPO 训练（在 V1 LoRA 基础上继续对齐执行正确性与简洁度）
from datasets import Dataset
from unsloth import PatchDPOTrainer
from trl import DPOConfig, DPOTrainer

PatchDPOTrainer()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{DRIVE_DIR}/models/v1_sft_lora",
    max_seq_length=CFG["model"]["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

def format_dpo_rows(pairs):
    rows = []
    for p in pairs:
        prompt_str = tokenizer.apply_chat_template(p["prompt"], tokenize=False, add_generation_prompt=True)
        chosen_str = p["chosen"][0]["content"] + tokenizer.eos_token
        rejected_str = p["rejected"][0]["content"] + tokenizer.eos_token
        rows.append({"prompt": prompt_str, "chosen": chosen_str, "rejected": rejected_str})
    return Dataset.from_list(rows)

dpo_ds = format_dpo_rows(dpo_pairs)
use_bf16 = torch.cuda.is_bf16_supported()

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=dpo_ds,
    args=DPOConfig(
        output_dir=f"{DRIVE_DIR}/checkpoints/v2_dpo",
        beta=dpo_cfg["beta"],
        learning_rate=dpo_cfg["learning_rate"],
        num_train_epochs=dpo_cfg["num_train_epochs"],
        per_device_train_batch_size=dpo_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=dpo_cfg["gradient_accumulation_steps"],
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=100,
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        max_length=CFG["model"]["max_seq_length"],
        max_prompt_length=1536,
        report_to="none",
    ),
)

dpo_trainer.train()

V2_LORA_DIR = f"{DRIVE_DIR}/models/v2_dpo_lora"
model.save_pretrained(V2_LORA_DIR)
tokenizer.save_pretrained(V2_LORA_DIR)
print("Saved V2 DPO adapter to:", V2_LORA_DIR)


In [ ]:
# 4. 最终验收对比（Base vs V1 SFT vs V2 DPO）并导出 V2 GGUF
v2_report = run_benchmark(f"{DRIVE_DIR}/models/v2_dpo_lora", eval_records, max_samples=200)

print("\n=== Final Benchmark: Base vs V1 SFT vs V2 DPO ===")
for k in ["json_valid_rate", "syntax_valid_rate", "execution_accuracy", "refusal_f1", "avg_output_chars", "avg_latency_ms"]:
    print(f"{k:22s} | Base: {str(base_report.get(k)):>8s} | V1 SFT: {str(v1_report.get(k)):>8s} | V2 DPO: {str(v2_report.get(k)):>8s}")

GGUF_V2_DIR = f"{DRIVE_DIR}/exports/v2_dpo_gguf"
os.makedirs(GGUF_V2_DIR, exist_ok=True)
for q in CFG["export"]["quantizations"]:
    model.save_pretrained_gguf(GGUF_V2_DIR, tokenizer, quantization_method=q)

!ls -lh {GGUF_V2_DIR}
